# 1D Gutzwiller Visualizer -- Development

**This is my development notebook -- production notebook is separate -- final code is `gviz.jl` -- which has changed relative to this notebook**

We will take $H=T+V+U$ with
\begin{align}
T &= \sum_{ij} -g_{ij} a_i^\dagger a_j\\
V&=\sum_i \delta_i a_i^\dagger a_i\\
U&=\frac{\eta}{2}\sum_j a_j^\dagger a_j^\dagger a_j a_j
\end{align}
and for now we will just hard-code nearest-neighbor 1D $g_{ij}=g_i \delta_{i,i_1}$.  We will want to have the freedom to make $g_i$ spatially dependent,  

Note, connecting with the more typical notation, $\eta=U$, and $g=t$.  

We make the ansatz
\begin{align}
|\psi\rangle = \prod_j \sum_n f_{jn} |n\rangle_j
\end{align}
so $f_{jn}$ is the amplitude for having $n$ particles on site $j$.  

We will store it as an `OffsetArray`.  The first index will be the site number (from 1 to N) and the second index the occupation number.

In [17]:
using OffsetArrays
using LaTeXStrings
using LinearAlgebra
using Random
using GLMakie, Makie.Colors
using Observables

In [2]:
testf=OffsetArray(zeros(10,3),0,-1)

10×3 OffsetArray(::Matrix{Float64}, 1:10, 0:2) with eltype Float64 with indices 1:10×0:2:
 0.0  0.0  0.0
 0.0  0.0  0.0
 0.0  0.0  0.0
 0.0  0.0  0.0
 0.0  0.0  0.0
 0.0  0.0  0.0
 0.0  0.0  0.0
 0.0  0.0  0.0
 0.0  0.0  0.0
 0.0  0.0  0.0

In [3]:
[j for j in axes(testf,2)]

3-element OffsetArray(::Vector{Int64}, 0:2) with eltype Int64 with indices 0:2:
 0
 1
 2

In [4]:
for j in 1:10
    testf[j,1]=1
end

In [5]:
[j for j in Iterators.drop(axes(testf,2),1)]

2-element Vector{Int64}:
 1
 2

$$
\psi_j=\sum_n \sqrt{n} f_{j,n-1}^* f_{jn}
$$

In [6]:
@doc raw"""
   randomf(numsites;minn=0,maxn=2,nbar=1)

Creates a random $f(j,n)$, with average of $\bar n$ 
particles per site
"""
function randomf(numsites;minn=0,maxn=2,nbar=1)
    f=OffsetArray(rand(ComplexF64,numsites,maxn-minn+1),0,minn-1)
    standarize!(f,nbar)
end

@doc raw"""
   uniff(numsites,fvals;minn=0)

Creates a uniform $f(j,n)$, where each site has the given
set of $f(n)$.  Change `minn` to set what the starting $n$ is.
"""
function uniff(numsites,fvals;minn=0)
    OffsetArray(stack(fvals for j in 1:numsites;dims=1),0,minn-1)
end

uniff

In [7]:
@doc raw"""
    nv(f)

gives the values of the $n$ coordinates in $f(j, n)$
"""
nv(f::OffsetArray)=axes(f,2)
@doc raw"""
    nv(f)

gives the values of the $n$ coordinates in $f(j, n)$, 
dropping the smallest
"""
rightnv(f)=Iterators.drop(nv(f),1)
@doc raw"""
    xv(f)

gives the values of the $j$ coordinates in $f(j, n)$
"""
xv(f::OffsetArray)=axes(f,1)
@doc raw"""
    rightxv(f)

gives the values of the $j$ coordinates in $f(j, n)$,
dropping the smallest
"""
rightxv(f)=Iterators.drop(xv(f),1)

@doc raw"""
    psi(f)

Calculates the order parameter $\psi_j=\sum_n \sqrt{n} f_{j,n-1}^* f_{jn}$.  
It assumes that $f$ is a `OffsetArray` where the allowed values of `n` are
set by the values of the keys.

also see `psi!(target,f)`.  See `uniff` to create a new $f$ with uniform entries.
"""
function psi(f)
    nvals = rightnv(f) #Iterators.drop(axes(f,2),1)
    xvals = xv(f) # =axes(f,1)
    result=[sum(sqrt(n)*f[j,n]*conj(f[j,n-1]) for n in nvals) for j in xvals]
end

@doc raw"""
    psi!(target,f)
Calculates the order parameter $\psi_j=\sum_n \sqrt{n} f_{j,n-1}^* f_{jn}$, and places
it in `target`.  It assumes that $f$ is a `OffsetArray` where the allowed values of `n` are
set by the values of the keys.

also see `psi(f)`.  See `uniff` to create a new $f$ with uniform entries.
"""
function psi!(target,f)
    nvals = rightnv(f) #Iterators.drop(axes(f,2),1)
    xvals=xv(f) #axes(f,1)
    for j in xvals
        target[j]=sum(sqrt(n)*f[j,n]*conj(f[j,n-1]) for n in nvals)
    end
    target
end



psi!

In [8]:
@doc raw"""
    density(f)

calculates the density $\rho_j=\sum_n n |f_{j,n}|^2$.  It assumes that $f$ is a `OffsetArray` where the allowed values of `n` are
set by the values of the keys.

see density!(target,f).
"""
function density(f)
    nvals = nv(f) #axes(f,2)
    xvals= xv(f) # axes(f,1)
    result=[sum(n*abs(f[j,n])^2 for n in nvals) for j in xvals]
end

@doc raw"""
    density!(target,f)

calculates the density $\rho_j=\sum_n n |f_{j,n}|^2$, placing it in `target`.  
It assumes that $f$ is a `OffsetArray` where the allowed values of `n` are
set by the values of the keys.

see density(f).
"""
function density!(target,f)
    nvals = nv(f) #axes(f,2)
    xvals= xv(f) #axes(f,1)
    for j in xvals
        target[j]=sum(n*abs(f[j,n])^2 for n in nvals)
    end
    target
end

density!

In [9]:
@doc raw"""
    normalize!(f)

rescales $f$ so that $\sum_n |f_{jn}|^2=1$ by taking $f_{jn}\to f_{jn}/\sqrt{\sum_m |f_{jm}|^2}$.

also see `standardize!` and `nproj!`
"""
function normalize!(f)
    nvals = nv(f) #axes(f,2)
    xvals= xv(f) #axes(f,1)
    for x in xvals
        n=norm(f[x,:])
        f[x,:]/=n
    end
    f
end

# Makes it so that the component labeled nbar has phase zero
@doc raw"""
    standarize!(f,n̄)

Rescales $f$ so that $\sum_n |f_{jn}|^2=1$ by taking $f_{jn}\to f_{jn}/\sqrt{\sum_m |f_{jm}|^2}$.
Also shifts the phase so that $f_{jn̄}$ is real.

also see `normalize!` and `nproj!`. 
"""
function standarize!(f,nbar)
    nvals = nv(f) #axes(f,2)
    xvals= xv(f) #axes(f,1)
    for x in xvals
        n=norm(f[x,:])
        f1=f[x,nbar]
        if abs(f1)>0
            phasefactor=f1/abs(f1)
            f[x,:]/=(n*phasefactor)
        else
            f[x,:]/=n
        end
    end
    f
end



standarize!

Our equations of motion come from
$$S=\int dt\, \langle \psi |i\partial_t|\psi\rangle - \langle \psi|H|\psi\rangle-\sum_j \lambda_j \sum_n |f_{jn}|^2 - \mu\sum_{jn} n |f_{jn}|^2$$
Where $\lambda_j$ and $\mu$ are Lagrange multipliers.  For each site we have a local constraint

$$\sum_n |f_{jn}|^2=1$$

One consequence is that we can make an arbitrary gauge transformation on each site

$$f_{jn}\to e^{i \theta_j} f_{j n}$$

The Lagrange Multiplier $\mu$ corresponds to number conservation

$$\sum_{jn} n |f_{jn}|^2=N$$

This gives us a global Gauge 

$$f_{jn}\to e^{i\theta n} f_{jn}$$

For small deviations, the projector onto the physical space takes the form

$$f_{jn}\to [\alpha_j + \beta (n-\bar n)]f_{jn}$$

** old writeup **

Given a set of $f_{jn}$'s, we want to be able to rescale it to have a fixed particle number.  Assuming we are doing this through some sort of dissipative chemical potential, we will have
\begin{align}
f_{jn}^\prime = \alpha_j+\beta (n-\bar n) f_{jn}
\end{align}
where $\alpha_j$ in general depends on the site, but $\beta$ does not.  (We have a global chemical potential only.  We will assume that $\beta$ is small, and $\alpha_j$ is near 1.

We define moments
\begin{align}
F_{js}&=\sum_n (n-\bar n)^s |f_{jn}|^2.
\end{align}
We want to choose $\alpha_j$ and $\beta$ so that
\begin{align}
F_{j0}^\prime&=1\\
&=\alpha_j^2 F_{j0}+ 2\alpha_j\beta F_{j1}+{\cal O}(\beta^2).
\end{align}
This can be solved for $\alpha_j$ as
\begin{align}
\alpha_j^2 &=\frac{1-2\beta F_{j1}/\sqrt{F_{j0}}}{F_{j0}}\\
\alpha_j &= \frac{1}{\sqrt{F_{j0}}}-\beta \frac{F_{j1}}{F_{j0}}
\end{align}


We then look at the number equation
\begin{align}
\sum_{j} F_{j1}^\prime &=0\\
&= \sum_j \alpha_j^2 F_{j1} +2\beta \alpha_j F_{j2}\\
&= \sum_j \frac{F_{j1}}{F_{j0}}\left(1-2\beta \frac{F_{j1}}{\sqrt{F_{j0}}}\right)+2\beta \frac{F_{j2}}{\sqrt{F_{j0}}}\\
&= \sum_j \frac{F_{j1}}{F_{j0}} -2\beta \sum_j \frac{1}{\sqrt{F_{j0}}}\left(\frac{F_{j1}^2}{F_{j0}}-F_{j2}\right)
\end{align}

solve that for $\beta$,
\begin{align}
\beta &=\frac{\sum_j \frac{F_{j1}}{F_{j0}}}{2\sum_j \frac{1}{\sqrt{F_{j0}}}\left(\frac{F_{j1}^2}{F_{j0}}-F_{j2}\right)}
\end{align}

We then normalize the f's

In [93]:
#rotate f so that the average occupation is given by targn
@doc raw"""
    nproj!(f,targn)
    nproj!(f,targn;nbar=nbar)

Projects $f$ into the physical space, where $\sum_n |f_{jn}|^2=1$
and the averag number of particles per site is `targn`.

If `nbar` is specified, then it also does a gauge transformation
so that the coefficient of $f_{j\bar n}$ is real, and
so that the order parameter is real on the central site.
"""
function nproj!(f,targn;threshold=1e-12,nbar=nothing)
    nvals = nv(f) #axes(f,2)
    xvals= xv(f) #axes(f,1)
    Ns=length(xvals)
    A=B=C=0.
    for x in xvals
        F0=F1=F2=F3=0.
        for n in nvals
            F0+= abs(f[x,n])^2
            F1+= (n-targn)*abs(f[x,n])^2
            F2+= (n-targn)^2*abs(f[x,n])^2
            F3+= (n-targn)^3*abs(f[x,n])^2
        end
        A+=F1/F0
        B+=(F1^2/F0-F2)/sqrt(F0)
        C+=F3-3*F2*F1/F0+2*F1^3/F0^2
    end
    beta=0.
    if abs(B)>threshold
        beta=A/(2*B) +A^2*C/(2*B)^3
    end
    alpha=1.
    for x in xvals
        if abs(beta)>threshold
            F0=F1=F2=0.
            for n in nvals
                 F0+= abs(f[x,n])^2
                 F1+= (n-targn)*abs(f[x,n])^2
                 F2+= (n-targn)^2*abs(f[x,n])^2
             end
             alpha=(-beta*F1 +sqrt(beta^2*F1^2+(1-beta^2*F2)*F0))/F0
        end
        normsq=0.
        for n in nvals   
            f[x,n]*= (alpha+beta*(n-targn))
            normsq += abs(f[x,n])^2
        end
        # Now force normalization -- since we have just been working to linear order in beta
        phasefactor=sqrt(normsq) # should be 1 if we did our math right
        if !(nbar isa Nothing) && abs(f[x,nbar])>threshold
            phasefactor*=f[x,nbar]/abs(f[x,nbar])
        end
        f[x,:]/=phasefactor
    end
    if !(nbar isa Nothing)
        setphase!(f,nbar)
    end
    f
end

nproj!

LoadError: UndefVarError: `setphase!` not defined

In [12]:
"""
    setphase!(f,nbar;threshold=1e-12)

Performs a gauge transformation, so that the order
parameter on the central site is real.
"""
function setphase!(f,nbar;threshold=1e-12)
    nvals = nv(f)#axes(f,2)
    xvals= xv(f)#axes(f,1)
    j=round(Int,(xvals[end]+xvals[1])/2)
    psi0=sum(sqrt(n)*f[j,n]*conj(f[j,n-1]) for n in Iterators.drop(nvals,1))
    if abs(psi0)<threshold
        return f
    end
    phi0=abs(psi0)/psi0
    for n in nvals
        if n==nbar
            continue
        end
        phasefactor=phi0^(n-nbar)
        for x in xvals
            f[x,n]*=phasefactor
        end
    end
    return f
end

setphase!

In [18]:
nproj!(uniff(5,ComplexF64[0.,1.,0.]),1;nbar=1)

5×3 OffsetArray(::Matrix{ComplexF64}, 1:5, 0:2) with eltype ComplexF64 with indices 1:5×0:2:
 0.0+0.0im  1.0+0.0im  0.0+0.0im
 0.0+0.0im  1.0+0.0im  0.0+0.0im
 0.0+0.0im  1.0+0.0im  0.0+0.0im
 0.0+0.0im  1.0+0.0im  0.0+0.0im
 0.0+0.0im  1.0+0.0im  0.0+0.0im

In [88]:
testf=OffsetArray(zeros(ComplexF64,10,3),0,-1)
for j in 1:10
    testf[j,1]=cos(0.4)*exp(2pi*im*rand())
    testf[j,0]=sin(0.4)*cos(pi/4+pi*(rand()-0.5))
    testf[j,2]=sin(0.4)*sin(pi/4+pi*(rand()-0.5))
end

In [89]:
sum(density(testf))

9.919244826593296

In [92]:
nproj!(testf,1;nbar=1)
sum(density(testf))

0.9999999999999999 0.9999999999999999 0.9999999999999998 1.0 0.9999999999999999 0.9999999999999998 0.9999999999999999 0.9999999999999999 1.0 0.9999999999999999 

10.000000000000002

The equations of motion are
\begin{align}
i e^{i\chi}\partial_t f_{jn}&= 
-\sum_i g_{ij} \left[ \psi_i^* \sqrt{n+1} f_{j,n+1} +\psi_i \sqrt{n} f_{j,n-1}\right]+ 
(\eta\, n(n-1)/2+\delta_j n) f_{jn}
\end{align}

In [17]:
# This is actually i df/dt, the way that I wrote it
function dfdt(f,p,t)
    g=p.g(t) #list of g's
    eta=p.eta(t) #list of eta's
    delta = p.delta(t) # list of deltas
    sym=p.sym(t)
    _dfdt(f,g,delta,eta,sym) # function barrier
end


function dfdt(f,p)
    g=p.g #list of g's
    eta=p.eta #list of eta's
    delta = p.delta # list of deltas
    sym=p.sym
    _dfdt(f,g,delta,eta,sym) # function barrier
end

function _dfdt(f,g,delta,eta,sym)
    ψ=psi(f)
    gpsi=similar(ψ)
    rightsites=Iterators.drop(axes(ψ,1),1)
    gpsi[0]=0
    for j in rightsites
        gpsi[j]=ψ[j-1]*g[j]
        gpsi[j-1]+=ψ[j]*g[j]
    end
    gpsi .+= sym
    result=similar(f)
    for j in axes(f,1)
        for n in axes(f,2)
            result[j,n]=f[j,n]*(-eta[j]*n*(n-1)/2+delta[j]*n)
        end
        for n in drop(axes(f,2),1)
            result[j,n]-= gpsi[j]*sqrt[n]*f[j,n-1]
            result[j,n-1]-= conj(gpsi[j])*sqrt[n]*f[j,n]
        end
    end
    return result
end 

_dfdt (generic function with 1 method)

uniff (generic function with 1 method)

In [16]:
f1=Observable(uniff(10,[0.0+0im,1.0+0im,0.0+0im]))

Observable(ComplexF64[0.0 + 0.0im 1.0 + 0.0im 0.0 + 0.0im; 0.0 + 0.0im 1.0 + 0.0im 0.0 + 0.0im; … ; 0.0 + 0.0im 1.0 + 0.0im 0.0 + 0.0im; 0.0 + 0.0im 1.0 + 0.0im 0.0 + 0.0im])


In [22]:
using Makie.Colors

In [23]:
HSV(0.1,1,1)

In [24]:
?HSV

search: HSV HSVA AHSV CholeskyPivoted



`HSV` is the Hue-Saturation-Value colorspace.


In [25]:
cf(val)= HSV(angle(val)*360/(2pi),abs(val)^2,1)

cf (generic function with 1 method)

In [26]:
angle(exp(im*pi/4)/(2pi))

0.7853981633974483

In [27]:
cf(exp(im*pi/4))

In [200]:
ComplexF64[1 for n in 1:3]

3-element Vector{ComplexF64}:
 1.0 + 0.0im
 1.0 + 0.0im
 1.0 + 0.0im

In [28]:
riffle(v1,v2)=collect(Iterators.flatten(zip(v1,v2)))
cf(val)= HSV(angle(val)*360/(2pi),abs(val)^2,1)

cf (generic function with 1 method)

In [65]:
function calcdelta!(deltas,disorder,disorderstrength,dw,t,dpattern,tphi)
    deltas .= disorder.*disorderstrength .+cos(dw*t+tphi).*dpattern
end

function calcgs!(gs,g,gw,t,gpattern,tphi)
    gs .= g.+cos(gw*t+tphi).*gpattern
end

function calcdelta(disorder,disorderstrength,dw,t,dpattern,tphi)
    deltas=similar(disorder)
    calcdelta!(deltas,disorder,disorderstrength,dw,t,dpattern,tphi)
    return deltas
end

function calcgs(g,gw,t,gpattern,tphi)
    gs=similar(gpattern)
    calcgs!(gs,g,gw,t,gpattern,tphi)
    return gs
end

calcgs (generic function with 1 method)

In [42]:
function gdftdt(f,p,t)
    g=calcgs(p.g,p.gw,t,p.gpattern,p.gtimephase)
    delta=calcdelta(p.disorder,p.disorderstrength,p.dw,t,p.dpattern,p.deltatimephase)
    eta=p.eta
    sym=p.sym
    df = _dfdt(f,g,delta,eta,sym)
    phasefactor=-im*exp(-im*p.disipationangle)
    df .*= phasefactor
    return df
end

gdftdt (generic function with 1 method)

In [43]:
function _dfdt(f,g,delta,eta,sym)
    ψ=psi(f)
    gpsi=similar(ψ)
    rightsites=Iterators.drop(axes(ψ,1),1)
    gpsi[1]=0
    for j in rightsites
        gpsi[j]=ψ[j-1]*g[j-1]
        gpsi[j-1]+=ψ[j]*g[j-1]
    end
    gpsi .+= sym
    result=similar(f)
    for j in axes(f,1)
        for n in axes(f,2)
            result[j,n]=f[j,n]*(eta*n*(n-1)/2+delta[j]*n)
        end
        for n in Iterators.drop(axes(f,2),1)
            result[j,n]-= gpsi[j]*sqrt(n)*f[j,n-1]
            result[j,n-1]-= conj(gpsi[j])*sqrt(n)*f[j,n]
        end
    end
    return result
end 

_dfdt (generic function with 1 method)

In [44]:
function rk4step(x,p,dxdt,t,deltat)
    k1=dxdt(x,p,t)
    k2=dxdt(x+k1*(deltat/2),p,t+deltat/2)
    k3=dxdt(x+k2*(deltat/2),p,t+deltat/2)
    k4=dxdt(x+k3*deltat,p,t+deltat)
    return x+(k1+2*k2+2*k3+k4)*(deltat/6)
end

rk4step (generic function with 1 method)

In [60]:
function gstep(f,p,dfdt,t,dt,targn,nbar)
    f=rk4step(f,p,gdftdt,t,dt)
    nproj!(f,targn;threshold=1e-12,nbar=nbar)
    return f
end

gstep (generic function with 1 method)

In [63]:
function gviz(;
        numsites=10, #number of sites
        maxn=2, #maximum number of particles per site
        nbar=1, #used for gauging wavefunction
        targn=1, #average number of particles per site)
        tsteprange=0.:0.0001:0.04,
        symrange=0.:0.001:0.1,
        disorderrange=0.:0.01:1.,
        disipationrange=0.:0.01:1,
        grange=0.0:0.01:1,
        ddrange=0.0:0.01:1,
        dfrange=0.0:0.01:2,
        dkrange=0:0.01:4,
        dprange=0.:0.01:2,
        gdrange=0.0:0.01:1,
        gfrange=0.0:0.01:2,
        gkrange=0:0.1:4,
        gprange=0.:0.01:2
        )
    f=uniff(numsites,ComplexF64[1 for n in 0:maxn])
    sites=xv(f)
    ns=nv(f)
    fo=Observable(f)
    #
    # Make Figure
    fig=Figure()
    display(fig)
    #
    # Make ax
    # -- middle panel, shows f
    ax=Axis(fig[2,1])
    rectangles=[Rect(i-0.5,j*0.15,1,0.1) for i in sites for j in ns]
    colors=@lift [cf($fo[i,n]) for i in sites for n in ns]
    poly!(rectangles;color=colors,strokecolor = :black,strokewidth = 1)
    #
    # Make ax2
    # Bottom panel, shows psisq and density
    ax2=Axis(fig[3,1],xlabel="x",ylabel=L"n\qquad|\psi|^2")
    dens=@lift density($fo)
    psisq=@lift abs.(psi($fo)).^2
    scatter!(ax2,dens)
    lines!(ax2,dens)
    scatter!(ax2,psisq)
    lines!(ax2,psisq)
    hidespines!(ax)
    hidedecorations!(ax)
    #
    # Make ax3
    deltas=[sin(2*pi*j/10) for j in sites]
    gs=[1. for j in sites[2:end]]
    deltao=Observable(deltas)
    gso=Observable(gs)
    deltasegs=@lift riffle($deltao,$deltao)
    sitesegs=riffle(sites.-0.5,sites.+0.5)
    ax3=Axis(fig[1,1],ylabel="drive")
    linesegments!(ax3,sitesegs,deltasegs;color=:black)
    bonds =sites[1:end-1].+0.5
    lines!(ax3,bonds,gso)
    scatter!(ax3,bonds,gso)
    #
    # Link Axes
    linkxaxes!(ax, ax2,ax3)
    #
    # Create reset buttons
    reset3=Button(fig[1,2],label="reset",tellheight=false)
    on(click->autolimits!(ax3),reset3.clicks) 
    reset2=Button(fig[3,2],label="reset",tellheight=false)
    on(click->autolimits!(ax2),reset2.clicks)  
    #
    # Sliders
    sliderbox=fig[4,1][1,1]
    sliders=SliderGrid(sliderbox,
        (label="timestep",range=tsteprange,format = "{:.4f}",startvalue=0.1),
        (label="symbreaking",range=symrange,format = "{:.3f}",startvalue=0.),
        (label="disorder",range=disorderrange,format = "{:.2f}",startvalue=0.),
        (label="disipation",range=disipationrange,format = "{:.2f}π/2",startvalue=0.),
        (label="g",range=grange,format = "{:.2f}",startvalue=1),
        (label="density drive",range=ddrange,format = "{:.2f}",startvalue=0.0),
        (label="density frequency",range=dfrange,format = "{:.2f}",startvalue=0.0),
        (label="density wave vector",range=dkrange,format = "{:.2f}",startvalue=0),
        (label="density phase",range=dprange,format = "{:.2f}π",startvalue=0.0),
        (label="hopping drive",range=gdrange,format = "{:.2f}",startvalue=0.0),
        (label="hopping frequency",range=gfrange,format = "{:.2f}",startvalue=0.0),
        (label="hopping wave vector",range=gkrange,startvalue=0),
        (label="hopping phase",range=gprange,format = "{:.2f}π",startvalue=0.0)
    )
    #unpack slider values
    tstepslider,symslider,dslider,disipationslider,gslider,ddriveslider,dwslider,dkslider,dphislider,gdriveslider,gwslider,gkslider,gphislider=sliders.sliders
    tstep=tstepslider.value
    sym=symslider.value
    disipationangle=@lift (pi/2)*$(disipationslider.value)
    disorder=Observable(rand(numsites).-0.5)
    disorderstrength=dslider.value
    ddrive=ddriveslider.value
    dk=dkslider.value
    dphi=@lift pi*$(dphislider.value)
    dw=dwslider.value
    gdrive=gdriveslider.value
    gw=gwslider.value
    gk=gkslider.value
    gphi=@lift pi*$(gphislider.value)
    g=gslider.value
    #
    # Create density pattern
    dpattern =@lift [$ddrive*sin(pi*$dk*j/(numsites+1)+$dphi) for j in 1:numsites]
    maxdelta=@lift ($disorder .*$disorderstrength .+$dpattern)
    mindelta=@lift ($disorder .*$disorderstrength .-$dpattern)
    lines!(ax3,sites,maxdelta;linewidth=0.5,color=:black)
    lines!(ax3,sites,mindelta;linewidth=0.5,color=:black)
    #
    t=Observable(0.)
    #
    # We need to be able to slide the "frequency"
    # sliders without making the quantities jump
    # wildly.  We do this by storing a temporal phase
    # offset
    #
    # Used global variable here before
    # during debugging may need to change
    # how information is moved around
    deltatimephase=Observable(0.)
    olddw=dw[]
    on(dw) do w
        deltatimephase[]-=(w-olddw)*t[]
        #global olddw=w
        olddw=w
    end
    Observables.@on begin
        calcdelta!(deltas,&disorder,&disorderstrength,&dw,&t,&dpattern,&deltatimephase)
        notify(deltao)
    end
    #
    gtimephase=Observable(0.)
    oldgw=gw[]
    on(gw) do w
        gtimephase[]-=(w-oldgw)*t[]
        #global oldgw=w
        oldgw=w
    end
    gpattern = @lift [$gdrive*sin(pi*$gk*j/(numsites)+$gphi) for j in 1:(numsites-1)]
    Observables.@on begin
        calcgs!(gs,&g,&gw,&t,&gpattern,&gtimephase)
        notify(gso)
    end
    gmax =@lift $g.+$gpattern
    gmin =@lift $g.-$gpattern
    lines!(ax3,bonds,gmax;linewidth=0.5,color=:blue)
    lines!(ax3,bonds,gmin;linewidth=0.5,color=:blue)
    #
    # 
    # Buttons
    buttonbox=fig[4,1][1,2]
    disorderbutton=Button(buttonbox[1,1];label="reset disorder")    
    on(disorderbutton.clicks) do clicks
        for j in 1:numsites
            disorder[][j]=rand()-0.5
        end
        notify(disorder)
    end
    isrunning = Observable(false)
    runlabel = map(cond -> cond ? "Stop" : "Run", isrunning)
    runbutton=Button(buttonbox[3,1];label=runlabel)
    # Package up parameters
    params=@lift (g=$g,gw=$gw,gpattern=$gpattern,disorder=$disorder,disorderstrength=$disorderstrength,
    dw=$dw,dpattern=$dpattern,eta=1,sym=$sym,disipationangle=$disipationangle,
    deltatimephase=$deltatimephase,gtimephase=$gtimephase)
    on(runbutton.clicks) do clicks; isrunning[] = !isrunning[]; end
    isrunning_notifier = Condition()
    on(cond -> cond && notify(isrunning_notifier), isrunning)
    errormonitor(@async while true
            if isrunning[]
                isopen(fig.scene) || break # stops if window is closed
                f=fo[]
                f=gstep(f,params[],gdftdt,t[],tstep[],targn,nbar)
                fo[]=f
                t[] += tstep[]               
                yield()
            else
                wait(isrunning_notifier)
            end
        end)
    return (f=fo,p=params)
end

gviz (generic function with 1 method)

In [64]:
gviz()

(f = Observable(ComplexF64[1.0 + 0.0im 1.0 + 0.0im 1.0 + 0.0im; 1.0 + 0.0im 1.0 + 0.0im 1.0 + 0.0im; … ; 1.0 + 0.0im 1.0 + 0.0im 1.0 + 0.0im; 1.0 + 0.0im 1.0 + 0.0im 1.0 + 0.0im]), p = Observable((g = 1.0, gw = 0.0, gpattern = [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], disorder = [0.42214740820126684, 0.22366119675179685, 0.36487811388525826, -0.2566687052759149, 0.46822896131908887, -0.07236636538499941, 0.48107351749301297, -0.17068847592854908, 0.4460041958007902, 0.1840459586937503], disorderstrength = 0.0, dw = 0.0, dpattern = [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], eta = 1, sym = 0.0, disipationangle = 0.0, deltatimephase = 0.0, gtimephase = 0.0)))

In [156]:
numsites=10
maxn=1
nbar=1
targn=0.5
#f=uniff(numsites,[0.0+0im,1.0+0im,0.0+0im])
f=uniff(numsites,[1.0/sqrt(2)+0im,1.0/sqrt(2)+0im])
sites=1:numsites
ns=0:maxn
fo=Observable(f)
fig=Figure()
display(fig)
ax=Axis(fig[2,1])
rectangles=[Rect(i-0.5,j*0.15,1,0.1) for i in sites for j in ns]
colors=@lift [cf($fo[i,n]) for i in sites for n in ns]
poly!(rectangles;color=colors,strokecolor = :black,strokewidth = 1)
ax2=Axis(fig[3,1],xlabel="x",ylabel=L"n\qquad|\psi|^2")
dens=@lift density($fo)
psisq=@lift abs.(psi($fo)).^2
scatter!(ax2,dens)
lines!(ax2,dens)
scatter!(ax2,psisq)
lines!(ax2,psisq)
hidespines!(ax)
hidedecorations!(ax)

false

riffle (generic function with 1 method)

In [157]:
deltas=[sin(2*pi*j/10) for j in sites]
gs=[1. for j in sites[2:end]]
deltao=Observable(deltas)
gso=Observable(gs)
deltasegs=@lift riffle($deltao,$deltao)
sitesegs=riffle(sites.-0.5,sites.+0.5)
ax3=Axis(fig[1,1],ylabel="drive")
linesegments!(ax3,sitesegs,deltasegs;color=:black)
bonds =sites[1:end-1].+0.5
lines!(ax3,bonds,gso)
scatter!(ax3,bonds,gso)

Scatter{Tuple{Vector{Point{2, Float64}}}}

In [158]:
linkxaxes!(ax, ax2,ax3)
reset3=Button(fig[1,2],label="reset",tellheight=false)
on(click->autolimits!(ax3),reset3.clicks) 
reset2=Button(fig[3,2],label="reset",tellheight=false)
on(click->autolimits!(ax2),reset2.clicks) 

ObserverFunction defined at In[158]:5 operating on Observable{Any}(0)

calcgs (generic function with 1 method)

In [161]:
deltatimephase[]

0.0

In [162]:
sliderbox=fig[4,1][1,1]
buttonbox=fig[4,1][1,2]
sliders=SliderGrid(sliderbox,
    (label="timestep",range=0.:0.0001:0.04,format = "{:.4f}",startvalue=0.1),
    (label="symbreaking",range=0.:0.001:0.1,format = "{:.3f}",startvalue=0.),
    (label="disorder",range=0.:0.01:1.,format = "{:.2f}",startvalue=0.),
    (label="disipation",range=0.:0.01*pi/2:pi/2,format = "{:.2f}",startvalue=0.),
    (label="g",range=0.0:0.01:1,format = "{:.2f}",startvalue=1),
    (label="density drive",range=0.0:0.01:1,format = "{:.2f}",startvalue=0.0),
    (label="density frequency",range=0.0:0.01:2,format = "{:.2f}",startvalue=0.0),
    (label="density wave vector",range=0:0.1:4,format = "{:.2f}",startvalue=0),
    (label="density phase",range=0.:2pi/60:2pi,format = "{:.2f}",startvalue=0.0),
    (label="hopping drive",range=0.0:0.01:1,format = "{:.2f}",startvalue=0.0),
    (label="hopping frequency",range=0.0:0.01:2,format = "{:.2f}",startvalue=0.0),
    (label="hopping wave vector",range=0:0.1:4,startvalue=0),
    (label="hopping phase",range=0.:2pi/60:2pi,format = "{:.2f}",startvalue=0.0)
)
tstepslider,symslider,dslider,disipationslider,gslider,ddriveslider,dwslider,dkslider,dphislider,gdriveslider,gwslider,gkslider,gphislider=sliders.sliders
tstep=tstepslider.value
sym=symslider.value
disipationangle=disipationslider.value
disorder=Observable(rand(numsites).-0.5)
disorderstrength=dslider.value
ddrive=ddriveslider.value
dk=dkslider.value
dphi=dphislider.value
dw=dwslider.value
dpattern =@lift [$ddrive*sin(pi*$dk*j/(numsites+1)+$dphi) for j in 1:numsites]
t=Observable(0.)
deltatimephase=Observable(0.)
olddw=dw[]
on(dw) do w
    deltatimephase[]-=(w-olddw)*t[]
    global olddw=w
end
Observables.@on begin
    #deltas .= &disorder.*&disorderstrength .+cos(&dw*&t)*&dpattern
    calcdelta!(deltas,&disorder,&disorderstrength,&dw,&t,&dpattern,&deltatimephase)
    notify(deltao)
end
disorderbutton=Button(buttonbox[1,1];label="reset disorder")    
on(disorderbutton.clicks) do clicks
    for j in 1:numsites
        disorder[][j]=rand()-0.5
    end
    notify(disorder)
end
isrunning = Observable(false)
runlabel = map(cond -> cond ? "Stop" : "Run", isrunning)
runbutton=Button(buttonbox[3,1];label=runlabel)

Button()

In [163]:
gdrive=gdriveslider.value
gw=gwslider.value
gk=gkslider.value
gphi=gphislider.value
g=gslider.value
gtimephase=Observable(0.)
oldgw=gw[]
on(gw) do w
    gtimephase[]-=(w-oldgw)*t[]
    global oldgw=w
end
gpattern = @lift [$gdrive*sin(pi*$gk*j/(numsites)+$gphi) for j in 1:(numsites-1)]
Observables.@on begin
    #gs .= $g.+cos(&gw*&t)*&gpattern
    calcgs!(gs,&g,&gw,&t,&gpattern,&gtimephase)
    notify(gso)
end

5-element Vector{ObserverFunction}:
 ObserverFunction defined at /Users/emueller/.julia/packages/Observables/YdEbO/src/Observables.jl:419 operating on Observable(0.0)
 ObserverFunction defined at /Users/emueller/.julia/packages/Observables/YdEbO/src/Observables.jl:419 operating on Observable{Any}(1.0)
 ObserverFunction defined at /Users/emueller/.julia/packages/Observables/YdEbO/src/Observables.jl:419 operating on Observable{Any}(0.0)
 ObserverFunction defined at /Users/emueller/.julia/packages/Observables/YdEbO/src/Observables.jl:419 operating on Observable(0.0)
 ObserverFunction defined at /Users/emueller/.julia/packages/Observables/YdEbO/src/Observables.jl:419 operating on Observable([0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0])

rk4step (generic function with 1 method)

_dfdt (generic function with 1 method)

gdftdt (generic function with 1 method)

In [167]:
params=@lift (g=$g,gw=$gw,gpattern=$gpattern,disorder=$disorder,disorderstrength=$disorderstrength,
    dw=$dw,dpattern=$dpattern,eta=1,sym=$sym,disipationangle=$disipationangle,
    deltatimephase=$deltatimephase,gtimephase=$gtimephase)

Observable((g = 1.0, gw = 0.0, gpattern = [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], disorder = [0.22296069670551588, -0.2505752894659691, -0.3759707420095676, 0.29550021537823845, 0.43820787997691557, 0.41014143151624294, -0.2660283304610286, -0.44805166815849407, 0.4749584036949198, 0.11540526321686839  …  -0.16336906074932467, -0.04045480668349821, -0.49774090207928834, -0.48940405572445034, 0.3605372426982201, -0.3060650520858148, -0.3993656856947092, 0.38957757867902976, -0.15037630667112756, 0.3860086688055723], disorderstrength = 0.0, dw = 0.0, dpattern = [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], eta = 1, sym = 0.0, disipationangle = 0.0, deltatimephase = 0.0, gtimephase = 0.0))


In [168]:
fo[]

100×2 OffsetArray(::Matrix{ComplexF64}, 1:100, 0:1) with eltype ComplexF64 with indices 1:100×0:1:
 0.707107+0.0im  0.707107+0.0im
 0.707107+0.0im  0.707107+0.0im
 0.707107+0.0im  0.707107+0.0im
 0.707107+0.0im  0.707107+0.0im
 0.707107+0.0im  0.707107+0.0im
 0.707107+0.0im  0.707107+0.0im
 0.707107+0.0im  0.707107+0.0im
 0.707107+0.0im  0.707107+0.0im
 0.707107+0.0im  0.707107+0.0im
 0.707107+0.0im  0.707107+0.0im
 0.707107+0.0im  0.707107+0.0im
 0.707107+0.0im  0.707107+0.0im
 0.707107+0.0im  0.707107+0.0im
         ⋮       
 0.707107+0.0im  0.707107+0.0im
 0.707107+0.0im  0.707107+0.0im
 0.707107+0.0im  0.707107+0.0im
 0.707107+0.0im  0.707107+0.0im
 0.707107+0.0im  0.707107+0.0im
 0.707107+0.0im  0.707107+0.0im
 0.707107+0.0im  0.707107+0.0im
 0.707107+0.0im  0.707107+0.0im
 0.707107+0.0im  0.707107+0.0im
 0.707107+0.0im  0.707107+0.0im
 0.707107+0.0im  0.707107+0.0im
 0.707107+0.0im  0.707107+0.0im

In [169]:
fo[]

100×2 OffsetArray(::Matrix{ComplexF64}, 1:100, 0:1) with eltype ComplexF64 with indices 1:100×0:1:
 0.707107+0.0im  0.707107+0.0im
 0.707107+0.0im  0.707107+0.0im
 0.707107+0.0im  0.707107+0.0im
 0.707107+0.0im  0.707107+0.0im
 0.707107+0.0im  0.707107+0.0im
 0.707107+0.0im  0.707107+0.0im
 0.707107+0.0im  0.707107+0.0im
 0.707107+0.0im  0.707107+0.0im
 0.707107+0.0im  0.707107+0.0im
 0.707107+0.0im  0.707107+0.0im
 0.707107+0.0im  0.707107+0.0im
 0.707107+0.0im  0.707107+0.0im
 0.707107+0.0im  0.707107+0.0im
         ⋮       
 0.707107+0.0im  0.707107+0.0im
 0.707107+0.0im  0.707107+0.0im
 0.707107+0.0im  0.707107+0.0im
 0.707107+0.0im  0.707107+0.0im
 0.707107+0.0im  0.707107+0.0im
 0.707107+0.0im  0.707107+0.0im
 0.707107+0.0im  0.707107+0.0im
 0.707107+0.0im  0.707107+0.0im
 0.707107+0.0im  0.707107+0.0im
 0.707107+0.0im  0.707107+0.0im
 0.707107+0.0im  0.707107+0.0im
 0.707107+0.0im  0.707107+0.0im

In [170]:
rk4step(testf,params[],gdftdt,t[],0.1)

10×3 OffsetArray(::Matrix{ComplexF64}, 1:10, 0:2) with eltype ComplexF64 with indices 1:10×0:2:
   0.027914+0.00794857im  0.949621-0.00444332im     -0.310232-0.0333801im
 -0.0417788+0.191342im    0.934299-0.0187949im       0.124046+0.270157im
    0.12251+0.283511im    0.897582-0.00847783im     -0.129457-0.286581im
 -0.0992616-0.184584im     0.96084+0.00224083im     0.0344122-0.17797im
   0.341206-0.00789747im  0.932395+0.0207982im       0.109027+0.0428584im
   0.273161-0.0935548im   0.941206+0.028971im        0.165363-0.0507775im
  0.0941889-0.21382im     0.969336+0.00897898im   -0.00829952+0.0751385im
   0.125902-0.298365im    0.936069-0.00105652im    0.00975506-0.13714im
   0.198174+0.00995952im  0.903216+0.000953624im    -0.380096-0.0188258im
    0.28806+0.0788959im   0.881575+0.00698358im     -0.328749-0.159683im

step (generic function with 2 methods)

In [172]:
nproj!(uniff(10,ComplexF64[1/sqrt(2),1/sqrt(2)]),0.5;threshold=1e-12,nbar=1)

10×2 OffsetArray(::Matrix{ComplexF64}, 1:10, 0:1) with eltype ComplexF64 with indices 1:10×0:1:
 0.707107+0.0im  0.707107+0.0im
 0.707107+0.0im  0.707107+0.0im
 0.707107+0.0im  0.707107+0.0im
 0.707107+0.0im  0.707107+0.0im
 0.707107+0.0im  0.707107+0.0im
 0.707107+0.0im  0.707107+0.0im
 0.707107+0.0im  0.707107+0.0im
 0.707107+0.0im  0.707107+0.0im
 0.707107+0.0im  0.707107+0.0im
 0.707107+0.0im  0.707107+0.0im

In [175]:
on(runbutton.clicks) do clicks; isrunning[] = !isrunning[]; end
isrunning_notifier = Condition()
    on(cond -> cond && notify(isrunning_notifier), isrunning)
        errormonitor(@async while true
        if isrunning[]
            isopen(fig.scene) || break # stops if window is closed
            f=fo[]
            f=step(f,params[],gdftdt,t[],tstep[],targn,nbar)
            #f=rk4step(f,params[],gdftdt,t[],tstep[])
            #nproj!(f,targn;threshold=1e-12,nbar=nbar)
            fo[]=f
            t[] += tstep[]
            yield()
            #step!(p)
            #sleep(p.delay) 
        else
            wait(isrunning_notifier)
        end
    end)

Task (runnable) @0x000000034ba27850

In [154]:
params[]

(g = 1.0, gw = 0.0, gpattern = [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], disorder = [0.3859131525532985, 0.33014343537959456, 0.4346404731660375, -0.34361572626500736, 0.47367803271356557, 0.20112159944175623, -0.1871177114931999, -0.41872655394417624, -0.2525625964606092, -0.27783379667472075  …  0.3462377761693407, -0.3158532603189419, -0.3523557732059853, 0.2911556122939446, -0.22584005655089245, -0.3047154883825449, -0.0065896623888309636, -0.28216840227458295, -0.048427682823765306, 0.28697608682084164], disorderstrength = 0.0, dw = 0.0, dpattern = [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], eta = 1, sym = 0.0, disipationangle = 0.0, deltatimephase = 0.0, gtimephase = 0.0)

In [155]:
density(fo[])

100-element OffsetArray(::Vector{Float64}, 1:100) with eltype Float64 with indices 1:100:
 0.9330127018922194
 0.9330127018922194
 0.9330127018922194
 0.9330127018922194
 0.9330127018922194
 0.9330127018922194
 0.9330127018922194
 0.9330127018922194
 0.9330127018922194
 0.9330127018922194
 0.9330127018922194
 0.9330127018922194
 0.9330127018922194
 ⋮
 0.9330127018922194
 0.9330127018922194
 0.9330127018922194
 0.9330127018922194
 0.9330127018922194
 0.9330127018922194
 0.9330127018922194
 0.9330127018922194
 0.9330127018922194
 0.9330127018922194
 0.9330127018922194
 0.9330127018922194